In [ ]:
#| default_exp diffusion

# Load deps

In [ ]:
# ! pip install -q torcheval

In [ ]:
# # # if src modules imported
# # from google.colab import drive
# # drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
import os, torch, math
import matplotlib as mpl
from torch.utils.data import DataLoader,default_collate
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from pathlib import Path
from torch.nn import init
from torch import nn
from datasets import load_dataset
from functools import partial
from torch.optim import lr_scheduler
from torch import optim
from fastprogress import progress_bar
from diffusers import UNet2DModel

from src.utils import set_seed
from src.datasets import inplace, DataLoaders, show_images
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB
from src.fid import ImageEval

# Config

In [ ]:
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
dataset_xl,dataset_yl = 'image','label'
dataset_name = "fashion_mnist"
bs = 512
lr = 4e-3
epochs = 10
models_path = Path("artifacts/models")
models_path.mkdir(exist_ok=True, parents=True)
os.environ['CUDA_VISIBLE_DEVICES']='1'
set_seed(42)
n_samples = bs

# Load dataset

In [ ]:
#|export
def abar(t): return (t*math.pi/2).cos()**2
def inv_abar(x): return x.sqrt().acos()*2/math.pi

def noisify(x0):
    device = x0.device
    n = len(x0)
    t = torch.rand(n,).to(x0).clamp(0,0.999)
    ε = torch.randn(x0.shape, device=device)
    abar_t = abar(t).reshape(-1, 1, 1, 1).to(device)
    xt = abar_t.sqrt()*x0 + (1-abar_t).sqrt()*ε
    return (xt, t.to(device)), ε

def collate_ddpm(b, xl): return noisify(default_collate(b)[xl])
def dl_ddpm(ds, ds_xl): return DataLoader(
            ds, batch_size=bs,
            collate_fn=partial(collate_ddpm, xl=ds_xl),
            num_workers=os.cpu_count()
)

In [ ]:
dsd = load_dataset(dataset_name)
@inplace
def transformi(b):
    b[dataset_xl] = [
        F.pad(TF.to_tensor(o), (2,2,2,2))-0.5
        for o in b[dataset_xl]
    ]

tds = dsd.with_transform(transformi)
dls = DataLoaders(
    dl_ddpm(tds['train'], dataset_xl),
    dl_ddpm(tds['test'], dataset_xl)
)
dl = dls.train

In [ ]:
(xt,t),eps = b = next(iter(dl))
show_images(xt[:16], imsize=1.5, titles=list(t[:16].cpu().numpy().round(3)));

# Train a new model with cosine scheduler

In [ ]:
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

In [ ]:
#|export
def init_ddpm(model):
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            if o.downsamplers:
                for p in list(o.downsamplers): init.orthogonal_(p.conv.weight)
    for o in model.up_blocks:
        for p in o.resnets: p.conv2.weight.data.zero_()
    model.conv_out.weight.data.zero_()

In [ ]:
opt_func = partial(optim.Adam, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [
    DeviceCB(), MixedPrecision(),
    ProgressCB(plot=True), MetricsCB(),
    BatchSchedCB(sched)
]
model = UNet(
    in_channels=1, out_channels=1,
    block_out_channels=(32, 64, 128, 256),
    norm_num_groups=8
)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

In [ ]:
# learn.fit(epochs)

# Save and load the checkpoint

In [ ]:
checkpoint_path = models_path / 'fashion_cos.pkl'

In [ ]:
# torch.save(learn.model, checkpoint_path)

In [ ]:
model = learn.model = torch.load(checkpoint_path, weights_only=False).cuda()

# Denoising
- this demonstrates a full (removeing all the noise) denoising step in the backward(denoising) process of diffusion
- the result is estimation of x_0 at different time steps

In [ ]:
def denoise(x_t, noise, t):
    device = x_t.device
    abar_t = abar(t).reshape(-1, 1, 1, 1).to(device)
    return ((x_t-(1-abar_t).sqrt()*noise) / abar_t.sqrt()).clamp(-1,1)

In [ ]:
show_images(xt[:16], imsize=1.5, titles=list(t[:16].cpu().numpy().round(3)));

In [ ]:
with torch.no_grad(): noise=learn.model((xt.cuda(),t.cuda()))
show_images(
    denoise(xt.cuda(),noise,t.cuda())[:16].clamp(-1,1),
    imsize=1.5,
    titles=list(t[:16].cpu().numpy().round(3))
);

# Sampling and measuring its performance

## Load the reference model

In [ ]:
from torch import nn
from src.init import GeneralRelu, init_weights
from functools import partial
from src.augment import get_dropmodel, Dropout
from src.augment import capture_preds
# TODO: we use this import to patch the capture_preds to the `Learner` class
# this is not a good practice at all. change it ASAP
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
cmodel = get_dropmodel(act_gr, norm=nn.BatchNorm2d, drop=0.1).apply(iw)
loaded_art = torch.load(models_path / 'data_aug2.pkl', weights_only=False)
cmodel.load_state_dict(loaded_art.state_dict())
cmodel = cmodel[:-3] # the desired feature head

In [ ]:
@inplace
def transformi(b): b[dataset_xl] = [F.pad(TF.to_tensor(o), (2,2,2,2))*2-1 for o in b[dataset_xl]]

tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=os.cpu_count())

dt = dls.train
xb,yb = next(iter(dt))

In [ ]:
ie = ImageEval(cmodel, dls, cbs=[DeviceCB()])

In [ ]:
def ddim_step(x_t, noise, abar_t, abar_t1, bbar_t, bbar_t1, eta, sig):
    sig = ((bbar_t1/bbar_t).sqrt() * (1-abar_t/abar_t1).sqrt()) * eta
    x_0_hat = ((x_t-(1-abar_t).sqrt()*noise) / abar_t.sqrt()).clamp(-1.5,1.5)
    if bbar_t1<=sig**2+0.01: sig=0.  # set to zero if very small or NaN
    x_t = abar_t1.sqrt()*x_0_hat + (bbar_t1-sig**2).sqrt()*noise
    x_t += sig * torch.randn(x_t.shape).to(x_t)
    return x_0_hat,x_t

@torch.no_grad()
def sample(f, model, sz, steps, eta=1.):
    ts = torch.linspace(1-1/steps,0,steps)
    x_t = torch.randn(sz).to(model.device)
    preds = []
    for i,t in enumerate(progress_bar(ts)):
        abar_t = abar(t)
        noise = model((x_t, t))
        abar_t1 = abar(t-1/steps) if t>=1/steps else torch.tensor(1)
        x_0_hat,x_t = f(x_t, noise, abar_t, abar_t1, 1-abar_t, 1-abar_t1, eta, 1-((i+1)/100))
        preds.append(x_0_hat.float().cpu())
    return preds

In [ ]:
sz = (n_samples,1,32,32)

In [ ]:
set_seed(42)
preds_100 = sample(ddim_step, model, sz, steps=100, eta=1.)
s = (preds_100[-1]*2)
# show_images(s[:25], imsize=1.5);
print(f"sampled images fid: {ie.fid(s)}, kid: {ie.kid(s)}")
print(f"real images fid: {ie.fid(xb)}, kid: {ie.kid(xb)}")

In [ ]:
set_seed(42)
preds_50 = sample(ddim_step, model, sz, steps=50, eta=1.)
s = (preds_50[-1]*2)
# show_images(s[:25], imsize=1.5);
print(f"sampled images fid: {ie.fid(s)}, kid: {ie.kid(s)}")
print(f"real images fid: {ie.fid(xb)}, kid: {ie.kid(xb)}")